# Introduction to GeoPandas for Working with Geospatial Vector Data (i.e., shapes: points, lines, and polygons)

<div class="alert alert-success">
    
## This notebook covers
- GeoPandas data structures
- Reading shapefiles and geodatabases
- Coordinate reference systems and reprojection
- GeoPandas attributes of shapes
- Constructing, clipping, and dissolving shapes
- Binary predicate functions
- Merging data
- Calculating distances
- Creating simple figures
</div>

<div class="alert alert-warning">

## Reminders

Remember, you can use Jupyter's built-in table of contents (hamburger on the far left) to jump from heading to heading.

---

This notebook will run in the MSUpy conda environment, which you created in the previous lesson. To select the Jupyter kernel associated with the MSUpy environment go to the Kernel tab, select Change Kernel, then in the pop up window drop down menu select the MSUpy kernel.

---

To turn on line numbers for code cells go to View menu and click Show Line Numbers.

</div>

# I. Importing Necessary Packages

In [ ]:
import pandas as pd
import geopandas as gpd
import pyogrio

<div class="alert alert-danger">
    
**Sidebar: GDAL and PROJ import errors**

You may see an error above about GDAL_DATA, or that you need to set a environmental variable for the GDAL package to work properly, or an error related to the proj package. If you see any errors or warning above, this notebook may run but yield incorrect results. Please post the error/warning you received in the pinned discussion for this lesson to get help resolving it before proceeding on with this notebook. If you don't see the error, great! 

</div>

# II. Why use Python GeoPandas for geospatial data analysis as opposed to ArcGIS or other GIS software?

If most or all of the data you work with are vector-based spatial data formats such as shapefiles, GeoJSON, or geodatabases, using GIS software will almost certainly be more desirable and efficient for you.

On the other hand, if most of the data you work with are not in these formats and your workflow is primarily in Python, GeoPandas may be the tool for you. For example, for a geoscientist (earth, environmental, climate sciences) who works most often with large multi-dimensional data arrays (time, level, lat, lon) in formats like .nc, .tif, .grib, .hdf, and .csv, Python will usually be much more efficient and reproducible for data analysis as compared to GIS. For this use case, packages like GeoPandas make working with the occasional shapefile or geodatabase very easy within a Python workflow instead of having to use a separate GIS tool for part of the analysis.

As we proceed through this notebook, we'll be looking at just of few of Python's geospatial analysis capabilities, mainly in the context that use of these tools is likely a minor part of a larger analysis using Python.

# III. Introduction to GeoPandas Data Structures

GeoPandas extends the Pandas package by adding support for geospatial vector data (i.e., shapes: points, lines, polygons). Under the hood, GeoPandas uses geometric operations from another Python package called [Shapely](https://shapely.readthedocs.io/en/stable/manual.html).

The core data structure in GeoPandas is the *GeoDataFrame*, which is similar to a Pandas DataFrame made up of an index and collection of *Series* (columns) as well as a special column named geometry. The geometry column is a *GeoSeries* which is a special structure that can handle shapes like points, lines, and polygons (a.k.a. geometries). Each shape object in a GeoSeries is a Shapely geometry object. A GeoDataFrame can contain as many Series and GeoSeries as you want, but only one GeoSeries (column of shape objects) can be activate at a time. We'll get into what that means later.        

Below is a schematic of what a GeoPandas GeoDataFrame looks like, where the green boxes indicate the index, yellow boxes indicate numerical or text data (with column names) in Series and pink boxes show a single GeoSeries which would contain shapes and have the column name geometry.


<img src="images/geodataframe.svg" alt="schematic of a dataframe" width="700"/> 
         
(Image Source: [GeoPandas Docs Getting Started Tutorial](https://geopandas.org/en/stable/getting_started/introduction.html#Concepts)

# IV. Loading a ShapeFile into a GeoDataFrame

Most of the time you'll probably start a geospatial analysis with a shapefile or geodatabase, both of which are comprised of multiple components. Three components are mandatory for shapefiles: 
- a main file that contains the feature geometry (.shp),
- an index file that stores the index of the feature geometry (.shx), and
- a dBASE table (.dbf) that stores the attribute information of the features.

We'll use multiple shapefiles (state boundaries and river centerlines), a geodatabase (EPA superfund sites), and a CSV file (species observations) as we learn about the functions and capabilities of GeoPandas.

First we'll read shapefiles containing the boundaries for a few US states and the centerlines for some large rivers into GeoDataFrames. The files we're using are subsets of data that are publicly available from the US Census Bureau and Natural Earth data project. If you're interested, the full data files are cb_2023_us_state_500k.shp from the [Census Bureau cartographic boundary files](https://www.census.gov/geographies/mapping-files/time-series/geo/cartographic-boundary.html) and ne_10m_rivers_lake_centerlines.shp, ne_10m_rivers_north_america.shp from the [Natural Earth data project](https://www.naturalearthdata.com/downloads/10m-physical-vectors/10m-rivers-lake-centerlines/).

```geopandas.read_file()``` can read almost any vector-based spatial data format.

In [ ]:
# load state boundaries from shapefile
gdf_states = gpd.read_file('data/admin_boundaries/state_boundaries/MS_LA_AR_AL_TN_cb_2023_us_state_500km.shp',
                          columns=['STUSPS','NAME','geometry'])
gdf_states

This is a GeoPandas GeoDataFrame and as you can see, it looks a lot like a Pandas DataFrame. Each row is a data entry for a different state. Notice the rightmost column named geometry. It contains geospatial shapes that represent the boundaries of each state. 

Let's look at the data type of each column.

In [ ]:
gdf_states.dtypes

The geometry column in a GeoDataFrame is data type geometry. This is indicating that the column named geometry is a GeoSeries that contains geospatial shapes.

Since there is only one column of type geometry, GeoPandas makes that column the *active geometry column*. You can have more than one GeoSeries column that contain shapes, but only one active at a time. By default, the active GeoSeries will be the one named geometry. 

Look once again at the geometry column of the GeoDataFrame. Notice the values inside the geospatial shapes. What are those numbers? Each set of numbers (separated by commas) is an x,y node (point) in the shape for each state. Large numbers, like we have, usually indicate that the data is in a coordinate reference system that has units of meters (as opposed to degrees latitude and longitude). We'll talk about coordinate reference systems next.

# V. Coordinate Reference Systems (CRS)

A *coordinate reference system* is a framework used to locate and map geographic features on the Earth's surface. The key components of a CRS are:

- **Coordinate System:** Defines the grid used to specify locations on the Earth, including the axes (like latitude/longitude or X/Y) and distance units of measurement (degrees or meters, for example). 

- **Datum:** The modeled version of the shape of the earth (usually represented by an ellipsoid). The datum also defines the origin (or reference point) of the coordinate system.

- **Projection:** A mathematical transformation that translates the spherical or ellipsoidal coordinates on the Earth's surface onto a flat surface like a map (Cartesian coordinates). No projection is perfect. Each will have different distortions which can affect area, shape, distance, and/or direction. Different projections are used based on the geographic area of interest and the purpose of the map. Some coordinate reference systems do not have a projection, meaning their coordinate system and datum define locations on a 3-dimensional Earth rather than a 2-dimensional map. 


As mentioned above, there are actually two types of CRS's: *geographic coordinate reference systems* and *projected coordinate reference systems*.

- **Geographic coordinate reference systems** are used to locate places on a 3-dimensional Earth surface based on two values, longitude and latitude, usually with units of decimal degrees. Units of angular distances are not linear due to the polar convergence of longitudes and the slight bulge of the earth at the equator. Therefore, a geographic CRS is not suitable to calculate or compare distances between locations. A common geographic CRS would be EPSG:4326, which uses a latitude and longitude coordinate system and the WGS84 datum (an ellipsoid) to represent locations on a 3-dimensional Earth. Confusingly, sometimes this CRS is referred to as the WGS84 CRS. Always using EPSG codes when discussing CRS's can eliminate a lot of confusion because they are unique identifiers. EPSG:4326 is the default CRS used in many GPS systems and is also used by Google Earth.

- **Projected coordinate reference systems** are based on a Cartesian coordinate system on a flat surface. Map projections are used to convert the 3D surface of the Earth into x and y coordinates of the projected CRS. A common projected CRS that you may encounter is EPSG:3857, also known as the Web Mercator projection. This CRS is used for many web mapping applications include Google Maps and Apple Map.


**Your choice of CRS will depend on how big your area of interest is, the region of the globe where it is located, and whether preserving shape, area, distance, or direction is most important to what you want to calculate.** Projections that preserve shape are called *conformal*; those that preserve area are called *equal-area*; those that preserve distance are called *equidistant*; and those that preserve direction are called *azimuthal*. These words might assist you in finding a projected CRS that preserves that geospatial properties most important to your analysis. Also, keep in mind that you can *reproject* from one CRS to another to increase the accuracy of calculations that require preservation of different geospatial properties (e.g. area vs distance).

**Where can you find information on different coordinate reference systems including the EPSG codes?** [https://spatialreference.org](https://spatialreference.org/), [https://epsg.io](https://epsg.io/), or simply type your CRS question into a web search engine and see what helpful websites pop up.


Let's see what CRS our state boundaries use:

In [ ]:
gdf_states.crs

The CRS of this data is EPSG:5070, which uses the NAD83 datum and an X/Y Cartesian coordinate system with units in meters. The information GeoPandas gives us let's us know that this CRS is equal-area and is appropriate for areas of interest in the contiguous United States (CONUS).

Now, let's plot our state shapes.

In [ ]:
# plot geospatial data from our GeoDataFrame
gdf_states.plot()

Notice the values on the X and Y axes. Those numbers are Cartesian X/Y values with units of meters as described in the CRS.

Also notice how we didn't specify which column of the ```gdf_states``` GeoDataFrame to plot. GeoPandas automatically plots the column named geometry because, by default, that column is the active GeoSeries. If we had multiple GeoSeries columns (containing geospatial shapes) we would need to specify the column name if we wanted to override the default plotting of the geometry column.

What if we wanted to plot our data such that the axes show latitude and longitude instead of meters? We could reproject to the geographic CRS EPSG:4326 for the plot like this: 

In [ ]:
# plot geospatial data from our GeoDataFrame in a new CRS
gdf_states.to_crs('EPSG:4326').plot()

Note that we didn't overwrite our GeoDataFrame with a new CRS here, we are simply reprojecting for the plot and not saving the result. You can double check this if you want by looking at the output of ```gdf_states.crs``` again, as we did earlier. We'll see how to reproject a GeoDataFrame *in place* (overwriting the data) a little later.

<div class="alert alert-info"> 
    
## Exercise 1: Read, check CRS, and plot shapefile data

In this exercise you'll practice using GeoPandas to read and display information from a shapefile.

Use GeoPandas to read the file ```data/admin_boundaries/state_boundaries/FL_GA_SC_NC_cb_2023_us_state_500k.shp``` into a GeoDataFrame called ```states``` and display the data rows.

</div>

In [ ]:
# add your code here
states = gpd.read_file('data/admin_boundaries/state_boundaries/FL_GA_SC_NC_cb_2023_us_state_500k.shp')
states

<div class="alert alert-info"> 

What is the CRS of the ```states``` GeoDataFrame and is it a geographic CRS or a projected CRS? 
</div>

In [ ]:
# add your code here
states.crs

Type answer here: The CRS is EPSG:4269 and it is a geographic CRS.

<div class="alert alert-info"> 

Make a simple plot of the state shapes in the ```states``` GeoDataFrame.
</div>

In [ ]:
# add your code here
states.plot()

<div class="alert alert-info"> 

Make a simple plot of the state shapes in the ```states``` GeoDataFrame in CRS EPSG:5070.
</div>

In [ ]:
# add your code here
states.to_crs('EPSG:5070').plot()

# VI. GeoSeries Attributes and General Methods (Functions)

There is a lot of information you can pull out of the shape objects in a GeoSeries. You can find the full list of [GeoSeries general methods and attributes in the GeoPandas Documentation](https://geopandas.org/en/stable/docs/reference/geoseries.html#general-methods-and-attributes). Let's look at a few of the available options that may be useful. These methods work on the entire GeoSeries (all the shapes at once) and some of them also work on individual shapes from a GeoSeries (a Shapely object).

First, let's look at a single shape object in the GeoSeries. We'll grab the Mississippi multipolygon and show that it is a Shapely geometry object.

In [ ]:
# reminder of what's in gdf_states
gdf_states

In [ ]:
# visual of a single shape in the GeoSeries
MS = gdf_states.loc[3,"geometry"]
MS

It should make sense now why this shape is a group of polygons called a multipolygon- all the little barrier islands!

In [ ]:
# showing this is a shapely geometry object
type(MS)

## Boundary

Gives you just the shape outline.

In [ ]:
MS.boundary

## Area

In [ ]:
MS.area

What units is this? Units are determined by the CRS. In this case the CRS units are meters, so the area is in square meters.

We can apply ```.area``` to the whole GeoDataFrame to get the area of each state.

In [ ]:
# areas for all the states in square km
gdf_states.area / 1E6 # m2 --> km2 conversion factor

Saving those results into a new column in the GeoDataFrame uses the same syntax we learned in the Pandas lesson.

In [ ]:
# save the result to a new column in the GeoDataFrame
gdf_states['AREA_KM2'] = gdf_states.area / 1E6
gdf_states

The data we are using thus far is in a projected CRS already (EPSG:5070 with units of meters), so our area calculations above did not cause an accuracy warning from the GeoPandas function ```.area```. See the sidebar below for more info.

<div class="alert alert-danger">

**Sidebar: not all CRS's are appropriate for area and distance calculations**

One very important thing to note is that many GeoPandas spatial functions are only accurate for shapes in a projected CRS. If you try to calculate something like area or create a new shape like a buffer with data in a geographic CRS, GeoPandas will warn you that you may not be getting accurate results. What you want to look for in your data's CRS is Cartesian units (like meters) to get accurate results. If your CRS has units of degrees, then your data is likely in a geographic CRS and you should reproject to a projected CRS before your calculations.
</div>

## Bounds of Shapes

Gives you the minx, miny, maxx, maxy bounds of the object in CRS units (here, meters)

In [ ]:
MS.bounds

Let's see how ```.bounds``` works on the entire GeoSeries.

In [ ]:
gdf_states.bounds

Let's look at some more attributes and functions that work on the full GeoSeries of shape objects

## Total Bounds of a GeoSeries

Gives you a single minx, miny, maxx, maxy bound of all the shapes in a GeoSeries combined 

In [ ]:
gdf_states.total_bounds

## Count Geometries in Multi-geometries

For multi-geometries, this counts how many shapes are within the multi-geometry shape (e.g. polygons in a multipolygon). This will return a value of 1 for non multi-geometries.

In [ ]:
gdf_states.count_geometries()

## Count Nodes in Shape

Gives you the number of nodes (points) that make up each shape

In [ ]:
gdf_states.count_coordinates()

## X/Y Coordinates of Shape Nodes

Pulls the x and y value of each node in each shape and puts them into separate x/y columns

In [ ]:
gdf_states.get_coordinates()

<div class="alert alert-info"> 

## Exercise 2: GeoSeries attributes

In this exercise you will use the ```states``` GeoDataFrame that you created in Exercise 1 and practice using GeoSeries attributes to get information about the shapes in the GeoDataFrame. 

First, get the Shapely geometry object for South Carolina from the GeoDataFrame into a variable called ```sc_shape```, display the Shapely object in your code cell output. Also, programmatically show that ```sc_shape``` is, in fact, a Shapely geometry object.
</div>

In [ ]:
# add your code here
sc_shape = states.loc[1,'geometry']
sc_shape

In [ ]:
# add your code here
type(sc_shape)

<div class="alert alert-info"> 

Get the minx, miny, maxx, maxy bounds only for the state of Florida and save the resulting GeoDataFrame to a new variable called ```fl_bounds```. Display the resulting row of data.
</div>

In [ ]:
# add your code here

fl_bounds = states.loc[states.STUSPS=='FL'].bounds
fl_bounds

<div class="alert alert-info"> 

Get only the miny value for the state of Florida as a single float data value, not a GeoSeries or GeoDataFrame object.
</div>

In [ ]:
# add your code here
fl_bounds.loc[2,'miny']

In [ ]:
fl_bounds.reset_index(drop=True).loc[0,'miny']

<div class="alert alert-info"> 

Reproject the ```states``` GeoDataFrame to EPSG:5070 and save the result in a new variable called ```states_5070```.
</div>

In [ ]:
# add your code here

states_5070 = states.to_crs('EPSG:5070')

states_5070.crs

<div class="alert alert-info"> 

Calculate the area of each state and save the result to a new column in ```states_5070``` called ```area```. Display all the rows in ```states_5070```. What are the units of the area column?
</div>

In [ ]:
# add your code here
states_5070['area'] = states_5070.area
states_5070

Type answer here: Units for areas in EPSG:5070 are square meters.

# VII. Showing Multiple GeoSeries on a Single Figure

Let's read another shapefile containing the centerlines of some large rivers and plot them along with the state boundaries in a single figure. The file below was created from ne_10m_rivers_lake_centerlines.shp and ne_10m_rivers_north_america.shp from the [Natural Earth data project](https://www.naturalearthdata.com/downloads/10m-physical-vectors/10m-rivers-lake-centerlines/).

In [ ]:
# get river data from shapefile into a GeoDataFrame
gdf_rivers = gpd.read_file("data/natural_earth/rivers/ms_major_rivers.shp")                    
gdf_rivers

<div class="alert alert-danger">

**Sidebar: Pay attention to projections** 

Unlike in GIS, there is no "on the fly" reprojection with GeoPandas or any other Python package. You need to be aware of the CRS's that your different datasets are using and make sure they match in order for the data to overlay correctly.
</div>

Let's check the CRS of ```gdf_rivers```.

In [ ]:
# check the crs
gdf_rivers.crs

Excellent, this data is also in EPSG:5070, meaning ```gdf_states``` and ```gdf_rivers``` should overlay correctly if we plot them together.

To plot more than one GeoSeries you must first create an Axes object which is like a base figure. Then, you can plot more GeoSeries on top of the base plot. For your convenience, here is the link to the API reference for [geopandas.GeoDataFrame.plot()](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.plot.html) (which is based on the matplotlib) where you can peruse all the different input parameter options.

In [ ]:
# create the base figure (Axes object) and plot the states on it
base = gdf_states.plot(facecolor='none',edgecolor='black',lw=0.5) 

# add the rivers on top of the base figure using the ax parameter
gdf_rivers.plot(ax=base)

# VIII. Loading Data from a Geodatabase into a GeoDataFrame

The geodatabase (.gdb) format is proprietary to ESRI, the developer of ArcGIS, however, GeoPandas ```.read_file()```is able to load data from a geodatabase. 

Let's load superfund site data from a geodatabase. Superfund sites are locations in the US where hazardous substances, pollutants, or contaminants are known or suspected to be released. The US Environmental Protection Agency (EPA) maintains the National Priorities List (NPL) of superfund sites. The NPL contains data that documents these sites and their clean up. The geodatabase we'll work with includes superfund sites that have undergone remediation and have been deleted or removed from the NPL, as well as active and proposed sites. 

The data file we will work with has been subset from a much larger publicly available data file by filtering layers and columns as well as subsetting to sites in Mississippi that have geospatial information. As of October 2024, [the larger data file can be obtained from data.gov](https://catalog.data.gov/dataset/npl-superfund-site-boundaries-epa9).   

In [ ]:
gdf_sites=gpd.read_file('data/EPA_superfund_boundaries/MS_EPA_NPL_Site_Boundaries.gdb')
gdf_sites.head()

This geodatabase is very simple, as it only contains one data layer of multipolygon objects, but this will rarely be the case when you are working with geodatabases. Before we move on, let's look at the tools you could use to see the names of all layers in a geodatabase and how you would read in a particular data layer if you had more than one. We need to import an additional package for this called pyogrio (we've already imported it in the beginning of our notebook).

In [ ]:
# use the pyogrio package to list all the layer names in a geodatabase
pyogrio.list_layers('data/EPA_superfund_boundaries/MS_EPA_NPL_Site_Boundaries.gdb')

This output is indicating that all the data in our geodatabase is stored in a single data layer called MS_EPA_NPL_Site_Boundaries and the layer's contents are multipolygons. If you are familiar with GIS, you know that there could be many more layer names in this list. If we had more layers, the ```pyogrio.list_layers()``` output might look like this:

```
array([['MS_EPA_NPL_Site_Boundaries', 'MultiPolygon'],
       ['some_other_layer_name', 'layer_geometry_type'],
       ['some_other_layer_name', 'layer_geometry_type']], dtype=object)
```


If there were more than one layer present in a geodatabase, we could load data from a specific layer into a GeoDataFrame by adding the layer parameter in ```gpd.read_file()```.

In [ ]:
gdf_sites=gpd.read_file('data/EPA_superfund_boundaries/MS_EPA_NPL_Site_Boundaries.gdb', layer='MS_EPA_NPL_Site_Boundaries')
gdf_sites

Moving on, let's check what CRS this data uses.

In [ ]:
gdf_sites.crs

It's in a different CRS! If we want to look at the locations of the superfund sites in relation to major rivers, we'll need all our data in the same CRS.

Let's reproject this data to EPSG:5070 to match our states and rivers data. We'll overwrite the current CRS and geometry column of ```gdf_sites``` using ```.to_crs()``` with the parameter ```inplace``` set to True. 

In [ ]:
gdf_sites.to_crs('epsg:5070',inplace=True)
gdf_sites.head()

Notice the nodes of each multipolygon have changed from units of decimal degrees to units of meters. Let's double check the new CRS info.

In [ ]:
gdf_sites.crs

Great! All our data now have the same EPSG:5070 CRS. 

Let's plot the state boundaries, rivers, and superfund sites on a single figure.

In [ ]:
# overlaying geometries from 3 different geodataframes
base = gdf_states.plot(facecolor='none',edgecolor='black',lw=0.5) # base figure (Axes)
gdf_rivers.plot(ax=base,color='blue')  # overlay on base figure
gdf_sites.plot(ax=base,color='orange') # overlay on base figure

What happened, why can't we see any of the orange multipolygon sites??

GeoPandas plotting is fairly simple, so unlike ArcGIS, we are unable to zoom in and out. If we wanted the capability to zoom in and out, we would have to use a fancier package for plotting (like folium). So the problem is that with the simple static plotting that GeoPandas offers, the sites are too small to see at this scale of the base plot. But, we can visualize the sites at this scale by plotting the sites as points instead of multipolygons. Points will automatically scale to be visible. We can plot the sites as points by finding the centroid of each site.

In [ ]:
# overlaying geometries from 3 different geodataframes, plotting sites as points
base = gdf_states.plot(facecolor='none',edgecolor='black')
gdf_rivers.plot(ax=base,color='blue')
gdf_sites.centroid.plot(ax=base,color='orange')

# IX. Geometric Constructive Functions

A geometric constructive function is a function that takes a GeoSeries of shape objects and creates new shape objects. We'll cover a few of them here but the full list of available functions can be found in the [Geometric Manipulations section of the GeoPandas User Guide](https://geopandas.org/en/stable/docs/user_guide/geometric_manipulations.html).

## Centroid 

We saw how to get the centroids of the superfund sites in the plot above. Let's save those centroids to a new column in the ```gdf_sites``` GeoDataFrame.

In [ ]:
# create a new GeoSeries in the GeoDataFrame that contains the centroid (point) of each multipolygon
gdf_sites['CENTROID']=gdf_sites.centroid
gdf_sites.head()

See how we can have more than one column that is a GeoSeries? The two columns of geospatial shapes are both GeoSeries. If we print the data types of all the columns we should see that we have two GeoSeries now. 

In [ ]:
# check out the data types
gdf_sites.dtypes

As mentioned earlier, the default behavior when plotting a GeoSeries in a GeoDataFrame is to plot whatever shapes are in the column named geometry. If you want to plot a different GeoSeries, you have to specify the column name like we do below with ```gdf_sites['centroid'].plot()```. 

The code below produces the same result as the previous figure that we created but there is a subtle difference. Below we are telling GeoPandas to plot the centroid column of ```gdf_sites```, whereas the similar line of code for the previous figure was telling GeoPandas to take the geometry column of ```gdf_sites```, calculate the centroids from the geometry and plot the centroids.

In [ ]:
# plotting a geoseries that is not named "geometry"
base = gdf_states.plot(facecolor='none',edgecolor='black')
gdf_rivers.plot(ax=base,color='blue')
gdf_sites['CENTROID'].plot(ax=base,color='orange')

Notice how nearly all the superfund sites are near major rivers or on the coast. That can't be good for water quality! 

Let's look at some more functions that may be helpful in a geospatial analysis.

## Buffer

First, let's make some simple buffers around our rivers. The [GeoPandas API reference](https://geopandas.org/en/stable/docs/reference.html) is where you should go to find out about all the parameters that you can input to a GeoPandas function. Here's the direct link to the page for [.buffer()](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoSeries.buffer.html).

In [ ]:
# 1000 meter buffer
gdf_rivers["buffer_1000m"] = gdf_rivers.buffer(1000)

# plot
base = gdf_states.plot(facecolor='none',edgecolor='black',figsize=(12,6)) # larger figure than default
gdf_sites.CENTROID.plot(ax=base,color='orange')
gdf_rivers.plot(ax=base,color='blue')
gdf_rivers.buffer_1000m.plot(ax=base,color='brown',alpha=0.25)

# change x and y extents
base.set_xlim(0.575E6,0.75E6)
base.set_ylim(.8E6,.9E6)

Notice the few things we added to our plotting code. We've: 
- changed the figure size with the ```figsize``` parameter in ```.plot()``` and
- changed the x and y extents of the plot by using functions ```.set_xlim()```, ```.set_ylim()``` on the base plot.

These changes result in a larger figure that is zoomed in on coastal Mississippi.

## Convex Hull
For a geometry with at least 3 points, the convex hull is the smallest convex Polygon containing all the points of the geometry. We'll create a convex hull polygon for each state and then plot the Mississippi state multipolygon and convex hull polygon on the same figure.

In [ ]:
# make a column containing the convex hull of each state
gdf_states['CONVEX_HULL']=gdf_states.convex_hull
gdf_states.head()

In [ ]:
# plot convex hull
base = gdf_states.loc[gdf_states.STUSPS=='MS'].plot(facecolor='none',edgecolor='black')
gdf_states.loc[gdf_states.STUSPS=='MS'].CONVEX_HULL.plot(ax=base,facecolor='yellow',edgecolor='orange',alpha=0.5)

## Envelope

An envelope is the smallest rectangular polygon (with sides parallel to the coordinate axes) that contains a geometry.

In [ ]:
# make a column containing the envelope of each state
gdf_states['ENVELOPE']=gdf_states.envelope
gdf_states.head()

In [ ]:
# plot the states and envelopes
base = gdf_states.plot(facecolor='none',edgecolor='black')
gdf_states.ENVELOPE.plot(ax=base,alpha=0.5)

## Union All

Returns a single shape (not a GeoSeries!) containing the union of all geometries in the GeoSeries.

In [ ]:
# get a single shape that is a combo of all the geometries in the geoseries
gdf_states.union_all()

In [ ]:
# what type of object is this
type(gdf_states.union_all())

<div class="alert alert-info"> 

## Exercise 3: Creating buffers

Create a 100m buffer around each superfund site and save the results to ```gdf_sites``` in a new column called BUFF_100M.

</div>

In [ ]:
gdf_sites['BUFF_100M']=gdf_sites.buffer(100)
gdf_sites.head()

<div class="alert alert-info"> 

Visualize the Chemfax site and site buffer. Make the site buffer your base plot and then overlay the site polygon in orange.
</div>

In [ ]:
# plot
base = gdf_sites.loc[[3]]['BUFF_100M'].plot()
gdf_sites.loc[[3]].plot(ax=base,color='orange')


# X. Binary Predicate Functions (True/False Spatial Queries)

These are functions that operate on a GeoSeries and return the result True or False for each shape in the GeoSeries. Most of these functions can be thought of as spatial queries, where we are querying aspects of the relationship between shape objects.

We'll cover only a few, but there are quite a few more GeoPandas functions like this (e.g., ```.overlaps()```, ```.touches()```, ```.covers()```, etc.). They are called binary predicates and can be found in the [GeoPandas API reference for Binary Predicates](https://geopandas.org/en/stable/docs/reference/geoseries.html#binary-predicates).

## Intersects

Let's look at how ```.intersects()``` works by selecting a single river and finding which states the river intersects with. The river we'll use is the Pearl River. First, we'll plot it.

In [ ]:
# plot the pearl river on the states
base = gdf_states.plot(facecolor='none',edgecolor='black')
gdf_rivers.loc[gdf_rivers.name=='Pearl'].plot(ax=base)

Notice how we used ```.loc[]``` with a condition inside it to locate the data row for the Pearl river in the GeoDataFrame. GeoPandas builds functionality on top of Pandas, so most of the things we learned in the Pandas lesson, like selecting with ```.loc[conditional expression]```, work with GeoPandas GeoDataFrames as well.

```.intersects()``` can test row by row between two GeoSeries of the same length or it can test each row (shape) in a GeoSeries against a single Shapely geometry object. We will test whether each state in the ```gdf_states``` GeoDataFrame intersects with the Pearl River Shapely geometry object. 

First, let's work out the syntax for selecting a single Shapely object from a GeoSeries when we don't know its row index. We need to select the Pearl River Shapely object which, if done correctly, should result in a Shapely geometry object, not a GeoSeries or GeoDataFrame. We can use a combination of ```.loc[]``` and ```.iloc[]```.

In [ ]:
# pull out the Shapely geometry object for the Pearl River
pearl_river = gdf_rivers.loc[gdf_rivers.name=='Pearl','geometry'].iloc[0]

# double check we have a shapely object, not a geodataframe or geoseries
type(pearl_river)

Now we can test whether the ```pearl_river``` object intersects with each state and return the result into a new column in the ```gdf_states``` called intersects_pearl.

In [ ]:
# new column with Boolean of whether each state intersects with the pearl river
gdf_states['INTERSECTS_PEARL']=gdf_states.intersects(pearl_river)
gdf_states

## Contains and Within

Now we'll find whether one shape contains another and similarly, whether one shape is within another. For example, polygon A *contains* point B, point B is *within* polygon A. Contains and within are really just two ways of finding the same information. 

We'll start simple. These first few examples test: 
- whether each shape in the GeoSeries of states contain one particular superfund site and
- whether one single state shape contains any of the superfund sites

In [ ]:
# select a single superfund site
one_site = gdf_sites.loc[0,"geometry"]
print(type(one_site))

# check if each state polygon contains that site
gdf_states.contains(one_site)

In [ ]:
# select a single state
one_state = gdf_states.geometry.iloc[0]  # this is Alabama
print(type(one_state))

# check if that state contains any of the superfund sites
one_state.contains(gdf_sites.geometry).any() # this is geopandas .any()

Notice the GeoPandas ```.any()``` function returns a NumPy boolean data type as opposed to the Python built-in boolean data type. We'll learn about the NumPy package and NumPy data types in the next lesson. NumPy boolean differs from Python built-in boolean in a few ways that make computation on arrays more efficient. Don't worry about the details yet though, you can interpret ```np.False_``` to mean ```False``` here. We could have also used the Python built-in ```any()``` function to accomplish the same thing as above, which would return the Python built-in boolean data type.   

In [ ]:
# we could have also used the Python built-in any function
any(one_state.contains(gdf_sites.geometry))

We can also ask the same question using within instead of contains.

In [ ]:
# get at the same information using within instead of contains

# check if any of the superfund sites are within Alabama
gdf_sites.geometry.within(one_state).any()

Now for a little added complexity! What if we want to check if each state in ```gdf_states``` contains any of the superfund sites? This would be the same thing we did above, but repeated for each state in ```gdf_states```. 

We will have to write a custom function for that and use ```.apply()```. GeoPandas ```.apply()``` works similarly to Pandas ```.apply()``` and lets us apply a custom function to each row of ```gdf_states```, checking whether each state contains any of the superfund sites. We already know that our superfund site data has been subset to Mississippi, so we expect a result of True for Mississippi and False for all other states. Here's how we can do it.

In [ ]:
# check each state to see if they contain any of the sites

# write a custom function that will apply to each row of a geodataframe
def state_contains_sites(row,sites):
    return row.geometry.contains(sites.geometry).any() 

# apply the function to each row of gdf_states with .apply()
# save the output to a new column
# axis=1 indicates that we want to apply the function to each row of gdf_states
gdf_states['HAS_SITES'] = gdf_states.apply(state_contains_sites,axis=1,args=(gdf_sites,))
gdf_states

```.apply()``` is like a loop, applying our function to 1 row of ```gdf_states``` at a time, moving row by row until the end of all the rows in the GeoDataFrame.


<div class="alert alert-info"> 

## Exercise 4: GeoPandas .within()

Use the example above of applying a custom function to a GeoDataFrame and test whether each superfund site CENTROID is within any of the 1000 meter river buffers (buffer_1000m column in ```gdf_rivers```). Save your result to a new column in ```gdf_sites``` called CEN_IN_RIVBUF. Show the CEN_IN_RIVBUF column results.



In [ ]:
# add your code here

def centroid_within_riverbuffer(row, rivers):
    return row.CENTROID.within(rivers.buffer_1000m).any()     

gdf_sites['CEN_IN_RIVBUF'] = gdf_sites.apply(centroid_within_riverbuffer, axis=1, args=(gdf_rivers,))
gdf_sites.CEN_IN_RIVBUF

We can see that even though the superfund centroid points we plotted looked very close to our major rivers, none of the centroids are within the 1000 meter river buffers. Although, parts of the superfund site polygons could be within the buffers. We'll check that later. 



# XI. Clip and Dissolve

## Clipping
We can use ```geopandas.clip()``` to clip shapes with other shapes. Let's clip the rivers to the rectangular envelope for Mississippi.

First, we'll select the Mississippi envelope object from the ```gdf_states``` GeoDataFrame.

In [ ]:
# select a single Shapely polygon (Mississippi envelope)
clip_object = gdf_states.loc[gdf_states.STUSPS=='MS','ENVELOPE'].iloc[0]
clip_object

Now we can clip the ```gdf_rivers``` GeoDataFrame with our clip object.

In [ ]:
# clip rivers with Mississippi envelope
clipped_rivers = gpd.clip(gdf_rivers, clip_object)

# plot states, Mississippi envelope, and clipped rivers
base = gdf_states.plot(facecolor='none',edgecolor='black')
gdf_states.loc[gdf_states.STUSPS=='MS','ENVELOPE'].boundary.plot(ax=base,color='red')
clipped_rivers.plot(ax=base,color='blue')

## Dissolving (Aggregating) Shapes

Let's look at how many repeated river names are in ```gdf_rivers```. It turns out that some of the rivers in the data are segmented, with different segments of the same river appearing in separate data rows. We can compare the number of rows in ```gdf_rivers``` to the number of unique river names:

In [ ]:
print(f'There are {gdf_rivers.shape[0]} rows in gdf_rivers, but only {len(gdf_rivers.name.unique())} unique river names')

Let's aggregate each group of river segments with the same name into a single shape. We can accomplish this with the ```.dissolve()``` function. 

In [ ]:
# dissolve segments of each river into a single shape object
gdf_rivers = gdf_rivers[['name','buffer_1000m','geometry']].dissolve(by='name',as_index=False,sort=False)

print(f'There are now {gdf_rivers.shape[0]} rows in gdf_rivers and {len(gdf_rivers.name.unique())} unique river names')

We've used ```.dissolve()``` here to stitch together river line segments but this function can also be used to aggregate other shapes, for example, dissolving county polygons into state
polygons.

What did GeoPandas do with those 1000m river buffers for all those rows that got dissolved? We can make a plot to see.

In [ ]:
# rivers in blue, buffers in red, blue+red=purple
base = gdf_states.plot(facecolor='none',edgecolor='black',lw=0.5)
gdf_rivers.buffer_1000m.plot(ax=base,color='red')
gdf_rivers.plot(ax=base,lw=0.5)

We can see purple where the red buffers and blue rivers overlap. From this plot, it's clear that some buffers got deleted. 

Should we have expected that? The answer is yes. With GeoPandas we can only use constructive and destructive functions on one GeoSeries at a time. Our dissolve operation worked only on the geometry column of ```gdf_rivers```, dissolving the river segments based on the river name and then dropping the full data row of each segment that was dissolved.

But that's ok we can just regenerate the buffers.

In [ ]:
gdf_rivers["buffer_1000m"] = gdf_rivers.buffer(1000)

# rivers in blue, buffers in red, blue+red=purple
base = gdf_states.plot(facecolor='none',edgecolor='black',lw=0.5)
gdf_rivers.buffer_1000m.plot(ax=base,color='red')
gdf_rivers.plot(ax=base,lw=0.5)

# XII. Distances

For simple distances, like if we want to find the distance between each site and a particular point we can use ```.distance()``` on a GeoDataFrame and give the distance function a Shapely point object. To calculate accurate distances though, you need all your shape objects to be in the same CRS and for that CRS to have cartesian units. Below, we'll find the distance between each site and the centroid of Mississippi. Both ```gdf_sites``` and ```gdf_states``` have the CRS EPSG:5070 which has units in meters.

In [ ]:
# distance between points
gdf_sites.distance(gdf_states.loc[gdf_states.NAME == "Mississippi"].centroid.iloc[0])

What are those units? Again, units are determined by the CRS so these distances are in meters. By the way, if we tried to calculate distances between objects in a geographic CRS with units in degrees, we'd receive a warning about inaccurate calculations.

If we want to do something a little more complex, like find the closest distance between each site polygon and the nearest river line, we can do a spatial join between ```gdf_sites``` and ```gdf_rivers```. For this particular application of finding the shortest distance between one shape (a site) and a group of other shapes (the rivers) there is a function called ```.sjoin_nearest()```.  

In [ ]:
sites_with_dist_to_river = gpd.sjoin_nearest(gdf_sites, gdf_rivers, how='left', rsuffix='rivers',distance_col='MIN_DIST', exclusive=True)
sites_with_dist_to_river

The results in the min_distance column indicate there are 3 sites that are closer than 1000m to a river centerline, meaning at least part of the site polygon falls within the 1000m river buffer.

Look at index 12. The closest part of the Rockwell International Wheel & Trim superfund polygon is only 528 meters from the Yalobusha River centerline. Let's make a zoomed in plot of this site's polygon and centroid, the Yalobusha River, and the 1000 meter buffer. 

In [ ]:
# plot Rockwell International polygon and centroid, Yalobusha River, and 1000 meter buffer

base = gdf_sites.loc[gdf_sites.SITE_NAME.str.contains('ROCKWELL')].boundary.plot(color='red',figsize=(8,8))
gdf_sites.loc[gdf_sites.SITE_NAME.str.contains('ROCKWELL')].centroid.plot(ax=base,color='orange')
gdf_rivers.loc[gdf_rivers.name=='Yalobusha'].plot(ax=base,color='blue')
gdf_rivers.loc[gdf_rivers.name=='Yalobusha','buffer_1000m'].plot(ax=base,color='brown',alpha=0.5)

# zoom in
base.set_ylim(1.209E6,1.213E6)
base.set_xlim(.568E6,.571E6)

Look at that! The centroid is not in the buffer, but part of the site polygon is. If we wanted to create a column telling us which site polygons lie at least partially within the river buffers we could use ```.intersects()``` for that.

<div class="alert alert-info"> 

## Exercise 5: Find sites that intersect a river buffer

Test whether each site polygon (geometry column in ```gdf_sites```) lies at least partially within any of the river buffers (buffer_1000m column in ```gdf_rivers```) and return your results to a new column in ```gdf_sites``` called ```INTERSECTS_RIVBUF```. You will need to write a new custom function that uses ```.intersects()``` and apply it to each row of the ```gdf_sites``` GeoDataFrame. 

</div>

In [ ]:
# add you code here

def site_intersects_riverbuffer(row, rivers):
    return row.geometry.intersects(rivers.buffer_1000m).any()

gdf_sites['INTERSECTS_RIVBUF'] = gdf_sites.apply(site_intersects_riverbuffer, axis=1, args=(gdf_rivers,))
gdf_sites

<div class="alert alert-info"> 

Use ```.loc[]``` to show only the data rows with sites that are at least partially in a 1000 meter river buffer.
</div>

In [ ]:
gdf_sites.loc[gdf_sites.INTERSECTS_RIVBUF==True]

<div class="alert alert-info"> 

Programatically find the integer number of sites that are at least partially in a 1000 meter river buffer and save the result to a new variable called ```nsites```.
</div>

In [ ]:
# add your code here
nsites = gdf_sites.INTERSECTS_RIVBUF.sum()
nsites

In [ ]:
nsites = gdf_sites.loc[gdf_sites.INTERSECTS_RIVBUF==True].shape[0]
nsites

# XIII. An iNaturalist Data Example

This example uses iNaturalist research grade quality observations of red eared sliders (Trachemys scripta elegans) in the US south from 2015-2024. The data was obtained online from the Global Biodiversity Information Facility https://doi.org/10.15468/dl.z2nq82 and subset to include only a few of the many available data columns.

We'll cover how to convert latitudes and longitudes from a CSV file into geometry objects, practice some data cleaning, and find the state and year with the most observations.

## Converting text coordinates to shapes (points)

Sometimes you'll have data points that are in a text format such as latitude and longitude values in a CSV file instead of a vector-based format like a shapefile. This is the case with iNaturalist data, which comes in tab delimited CSV files. Pandas and GeoPandas make it easy to read data from a CSV file into a Pandas DataFrame and convert to a GeoDataFrame by creating point geometries from the latitude and longitude data values. 

We'll start by reading the data file into a DataFrame with Pandas.

In [ ]:
# read only specific columns of the csv file
turtles_df = pd.read_csv('data/GBIF/redEaredSliders_iNaturalist_GBIF_USsouth_2015-2024.csv')
turtles_df.head()

To turn the Pandas DataFrame above into a GeoPandas GeoDataFrame we need two things: a geometry column of shapes and a coordinate reference system (CRS).

Now, we don't know exactly how all those latitudes and longitudes were collected, but the way iNaturalist works is users take pictures of specimens and upload them to iNaturalist. We'll assume that most users do this with the iNaturalist app on their cell phones. Cell phones tag photos with GPS locations in the geographic CRS EPSG:4326 (not projected), so that is the CRS we'll use for this data.

Let's make ESPG:4326 points from the latitude and longitude data values. GeoPandas has a convenient function for this.

In [ ]:
# generate vector data points
geometry = gpd.points_from_xy(turtles_df['decimalLongitude'], turtles_df['decimalLatitude'], crs='EPSG:4326')
geometry

Now we can create a GeoDataFrame from the DataFrame and geometry information.

In [ ]:
# convert pandas dataframe to a geopandas geodataframe
turtles_gdf = gpd.GeoDataFrame(turtles_df, geometry=geometry)

print(turtles_gdf.crs)
turtles_gdf.head()

## More GeoPandas practice with the iNaturalist data

Let's get some more GeoPandas practice by working with this GeoDataFrame to do the following:

- drop rows that are missing any of the following info: stateProvince, decimalLatitude, decimalLongitude, coordinateUncertaintyInMeters, or eventDate
- drop rows where the coordinateUncertaintyInMeters column is greater than 150km
- drop all data that are not within our ```gdf_states```
- see if there are any discrepancies between the stateProvince value and the latitude/longitude point based on our ```gdf_state``` boundaries
- find the state with the most observations of red eared sliders
- make a cloropleth map where we color the states by total observations
- find the year with the most observations of red eared sliders

Let's start by taking a look at the turtles_gdf info.

In [ ]:
# look at shape, dtypes, non-null, and header
print(turtles_gdf.info())

We can see that the two text columns and the dates are strings (object), the columns that look like numbers have been assigned a numerical data type (float), and the geometric points are the geometry data type. We'll convert the dates to datetimes later. 

It looks like we have missing data in the stateProvince and coordinateUncertaintyInMeters columns, but no missing values in the other columns. 

### drop rows with Nan

Let's drop all the rows with any missing values using ```.dropna()```.

In [ ]:
# drop rows with Nan
turtles_filtered = turtles_gdf.dropna(how='any')
turtles_filtered.shape

### drop rows where coordinate uncertainty isn't very precise

We're arbitrarily choosing an coordinate uncertainty of 150 km. 

In [ ]:
# drop rows with high uncertainty in the geospatial location
turtles_filtered = turtles_filtered.loc[turtles_filtered.coordinateUncertaintyInMeters < 150000]
turtles_filtered.shape

### drop rows outside of our gdf_states

Now dropping all rows with data points beyond the boundaries of the states in ```gdf_states```. We could approach this a few different ways. 
- One way would be to clip ```turtles_filtered``` with the union of the state shapes in ```gdf_states```.
- Another way would be to drop rows from ```turtles_filtered``` with stateProvince values that don't match any of the values from the NAME column of ```gdf_states```.

These two methods could potentially produce different results, for example, if an observation was recorded in Mississippi but the lat/lon point recorded is beyond the state boundary in the Gulf of Mexico.  

We'll start with the clipping approach. If you remember, our ```gdf_states``` has a CRS of EPSG:5070, so we'll reproject it to match our turtle data.

In [ ]:
# first convert the gdf_states to the same crs as the turtle data
gdf_states_4326 = gdf_states.to_crs('EPSG:4326')

# create a clip object
clip_object = gdf_states_4326.union_all()
clip_object

In [ ]:
# clip the turtle data
turtles_filtered = gpd.clip(turtles_filtered,clip_object)
print(turtles_filtered.shape)

We'll now check that the only states remaining in ```turtles_filtered.stateProvince```are the 5 states in ```gdf_states```. 

In [ ]:
# these are the 5 states we're looking for
print(gdf_states.NAME)

# these are the remaining states in the turtle data
turtles_filtered.stateProvince.unique()

Looks like there's at least 1 data point with a stateProvince of Missouri. Let's look at that row(s) on a map.

In [ ]:
# overlay the two geodataframes on a plot
base = gdf_states_4326.plot(facecolor='none',edgecolor='black',lw=0.5) # base figure (Axes)
turtles_filtered.loc[turtles_filtered.stateProvince=='Missouri'].plot(ax=base,color='magenta',markersize=1)  # overlay on base figure

It's just one point that is located in Arkansas near the border with Missouri. We don't know which is correct, the stateProvince or the location of the point so we'll drop this row.

In [ ]:
# drop rows where stateProvince isn't one of the 5 states in gdf_states
turtles_filtered = turtles_filtered.loc[turtles_filtered.stateProvince.isin(gdf_states.NAME)]
turtles_filtered.shape

Visualizing the remaining data

In [ ]:
# overlay the two geodataframes on a plot
base = gdf_states_4326.plot(facecolor='none',edgecolor='black',lw=0.5) # base figure (Axes)
turtles_filtered.plot(ax=base,color='blue',markersize=1)  # overlay on base figure

### find state discrepancies

Before we start counting how many observations there are per state, let's check for any more discrepancies between the stateProvince value and the latitude/longitude point based on our ```gdf_state``` boundaries. For example, if a record in ```turtles_filtered``` says its stateProvince is Mississippi but the geometry Point is actually in Alabama, we'll drop those rows.

In [ ]:
# check if turtle points are within the boundaries of the state in the stateProvince column
# we'll use .apply to apply this to the turtle data

def point_within_state(row, states):
    # get the state shape that matches the turtle stateProvince column
    one_state_poly = states.loc[states.NAME == row.stateProvince,'geometry'].iloc[0]
    return row.geometry.within(one_state_poly) # test if each turtle point is within the state boundary

In [ ]:
# apply the function
turtles_filtered['pointInState'] = turtles_filtered.apply(point_within_state, axis=1, args=(gdf_states_4326,))
turtles_filtered.head()

In the results returned to the column pointInState, a value of True means the point does fall in the state that appears in the stateProvince column and a value of False means that the state the point is within and the state that appears in the stateProvince column are not the same. Remember that the boolean value True is equal to 1 and False is equal to zero. If we sum all the values in the pointInState column the result would be how many True values are present. We can use bitwise not to sum the False values.

In [ ]:
# how many False values are in the pointInState column
(~turtles_filtered.pointInState).sum()

There is 1 row where the state and lat/lon point location don't match. Let's look at the stateProvince for that point and plot the lat/lon point on a map.

In [ ]:
# get stateProvince value of the row with the discrepancy
turtles_filtered.loc[turtles_filtered.pointInState == False,'stateProvince']

In [ ]:
# plot discrepancy
base = gdf_states_4326.plot(facecolor='none',edgecolor='black',lw=0.5) # base figure (Axes)
turtles_filtered.loc[turtles_filtered.pointInState == False].plot(ax=base,color='magenta',markersize=1)  # overlay on base figure

The stateProvince for this point is Louisiana but the point appears in Arkansas. We don't know which is correct so we'll drop this row. The condition inside ```.loc[]``` in the code below means keep rows where point_in_state is True.

In [ ]:
# drop discrepancy
turtles_filtered = turtles_filtered.loc[turtles_filtered.pointInState]
turtles_filtered.shape

### aggregate points by state to find the max

Our data is now clean enough to sum the points per state. The easiest way to do this would be to count how many of each state there are in the stateProvince column

In [ ]:
# sum base on the stateProvince value
turtles_filtered.stateProvince.value_counts()

Louisiana has the most observations of red-eared sliders.

Imagine if we didn't have the stateProvince column though and we had to count how many turtle points are within each state shape. We could do that with a spatial join and then count how many of each state are in the NAME column.

In [ ]:
# spatial join the gdf_states to the turtle data
joined_gdf = gpd.sjoin(turtles_filtered,gdf_states_4326,how='inner',predicate='within')
joined_gdf.head()

In [ ]:
# now sum based on the NAME column value
joined_gdf.NAME.value_counts()

By the way to return only the name of the state with the maximum value count we can use ```.idxmax()```

In [ ]:
joined_gdf.NAME.value_counts().idxmax()

We can programatically assert that both methods produced the same result.

In [ ]:
# assert the results of both methods are identical
assert all(turtles_filtered.stateProvince.value_counts() == joined_gdf.NAME.value_counts()), 'value count methods are not the same'

Remember, no output from an assert statement means the condition is True. In this case, the result of our first value count is the same as the result of our second value count.

### aggregate points by year to find the max

The easiest way to count data observations per year would be to use ```.resample()``` on datetime objects. When we read the data file into a DataFrame, Pandas assigned the eventDate column to the object dtype, so we'll need to convert this column to datetimes.

In [ ]:
# create datetimes
turtles_filtered['dateTime'] = pd.to_datetime(turtles_filtered['eventDate'], format='ISO8601',utc=True)
print(turtles_filtered.eventDate.dtype)
print(turtles_filtered.dateTime.dtype)

Now we can time index the turtle data by using the dateTime column values as the index.

In [ ]:
# set the geodataframe index to the datetimes
turtles_filtered = turtles_filtered.set_index(turtles_filtered.dateTime)
turtles_filtered.head()

In [ ]:
# use resample to group by years and count how many points are in each group
turtles_filtered.geometry.resample('YE').count()

The year with the highest number of observations is 2021. 

By the way, to return the year with the most observations as an integer we can use a combination of ```.idxmax()``` and ```.year```.

In [ ]:
turtles_filtered.geometry.resample('YE').count().idxmax().year

<div class="alert alert-info"> 

# XIV. Exercise: Putting It All Together

Use GeoPandas to read, manipulate, analyze, and visualize data from shapefiles of world countries and cities. 

## A) Read a shapefile into a GeoDataFrame and visualize

### read shapefile

The file ```data/admin_boundaries/county_boundaries/tl_2020_us_county_subset.shp``` contains the county boundaries in five states (MS, LA, AR, AL, TN) from the US Census Bureau. Load it into in a GeoDataFrame called ```counties```. Print a preview of the rows in ```counties```.

In [ ]:
counties = gpd.read_file('data/admin_boundaries/county_boundaries/tl_2020_us_county_subset.shp')
counties.head()

<div class="alert alert-info"> 

How many data rows are in ```counties```?

In [ ]:
# add your code here
counties.shape

Type your answer: 383 data rows

<div class="alert alert-info"> 

Are the columns in ```counties``` numeric or non-numeric data types?

In [ ]:
# add your code here
counties.dtypes

Type your answer: all columns are non-numeric

<div class="alert alert-info"> 

What is the CRS of ```counties``` and what are the units? Is it a geographic or projected CRS?

In [ ]:
# add your code here
counties.crs

Type your answer: EPSG 4269 with units in degrees, which is a geographic CRS

<div class="alert alert-info"> 

### visualize polygons

Make a simple plot of ```counties```.

In [ ]:
counties.plot()

<div class="alert alert-info"> 

## B) Create a GeoDataFrame from a .txt file and visualize

### read data from a text file

The file ```data/population/us_cb_2020_counties/centers_of_population_by_county.txt``` contains the geographic center of population for each county in the US. The data in the file is comma delimited. Read it into a Pandas DataFrame called ```pop_df```. Print a preview of data rows.

In [ ]:
pop_df = pd.read_csv('data/population/us_cb_2020_counties/centers_of_population_by_county.txt')
pop_df.head()

<div class="alert alert-info"> 

How many data rows are in ```pop_df```?

In [ ]:
# add your code here
pop_df.shape

Type your answer: 3221 data rows

<div class="alert alert-info"> 

Are the columns in ```pop_df``` numeric or non-numeric data types?

In [ ]:
# add your code here
pop_df.dtypes

Type your answer: all numeric except COUNAME and STNAME which are non-numeric

<div class="alert alert-info"> 

### generate point shapes from latitudes and longitudes

Use GeoPandas to generate points from the LATITUDE and LONGITUDE columns in ```pop_df```. Use the same CRS as the ```counties``` GeoDataFrame. Save your points in a variable called ```geometry```.

In [ ]:
# generate vector data points
geometry = gpd.points_from_xy(pop_df['LONGITUDE'], pop_df['LATITUDE'], crs=counties.crs)
geometry

<div class="alert alert-info"> 

### convert DataFrame to GeoDataFrame

Use your variables ```pop_df``` and ```geometry``` to create a GeoDataFrame called ```pop_gdf```. Print the CRS, column data types, and a preview of data rows in ```pop_gdf```.

In [ ]:
pop_gdf = gpd.GeoDataFrame(pop_df, geometry=geometry)

print(pop_gdf.crs)
print(pop_gdf.dtypes)
pop_gdf.head()

<div class="alert alert-info"> 

### visualize points 

Make a simple plot of the population center points in ```pop_gdf```. This plot doesn't need to look nice, the goal is simply to see the spatial extent of the points.

In [ ]:
pop_gdf.plot()

<div class="alert alert-info"> 

## C) Reproject GeoDataFrames

We'll be making buffers later in the exercise, therefore we will reproject to a projected CRS with cartesian coordinates. Reproject ```counties``` and ```pop_gdf``` to EPSG 5070.

In [ ]:
counties = counties.to_crs('EPSG:5070')
pop_gdf = pop_gdf.to_crs('EPSG:5070')

print(counties.crs)
pop_gdf.crs

<div class="alert alert-info"> 

What units does EPSG 5070 have?

Type your answer: meters

<div class="alert alert-info"> 

## D) Perform a spatial join on two GeoDataFrames and visualize

### spatial join

The goal is to join the population and geometry information from ```pop_gdf``` to ```counties```.

Start by copying the geometry in ```pop_gdf``` to new column in ```pop_gdf``` called POP_CENTER. 

In [ ]:
pop_gdf['POP_CENTER'] = pop_gdf['geometry']
pop_gdf.columns

<div class="alert alert-info"> 

Now use GeoPandas to join only the STNAME, POPULATION, POP_CENTER, and geometry columns from ```pop_gdf``` to ```counties``` where the geometries intersect. Save the result to a new variable called ```county_pop```.

**Hints:** This is a ```geopandas.sjoin()``` procedure with ```how='inner'```. Use the double plain bracket syntax to select specific columns from ```pop_gdf``` in the join. There should be 383 rows in your result.

In [ ]:
county_pop = gpd.sjoin(counties,pop_gdf[['STNAME','POPULATION','POP_CENTER','geometry']],how='inner')
county_pop

<div class="alert alert-info"> 

### visualize population by county

Make a map where counties are colored by population (cloropleth map). Include a horizontal colorbar labeled "Population in 2020").

In [ ]:
county_pop.plot(column='POPULATION', legend=True,
                legend_kwds={"label": "Population in 2020", "orientation": "horizontal"})

<div class="alert alert-info"> 

### visualize county boundaries and population centers

Using your ```county_pop``` GeoDataFrame, make a plot with the county boundaries in black and the population centers in blue. Do not fill the counties (only use black borders) and reduce the thickness of the borders with the parameter ```lw=0.5```. For the population centers use the parameter ```marker='.'```.

In [ ]:
base = county_pop.plot(facecolor='none', edgecolor='black',lw=0.5)
county_pop.POP_CENTER.plot(ax=base, marker='.')

<div class="alert alert-info"> 

## E) Use dissolve to find state population

Use the columns STNAME, POPULATION, and geometry in ```county_pop``` to dissolve the county shapes into state shapes and sum the population by state. Save the result to a new variable called ```state_pop```.

**Hint:** Summing a column during a dissolve is something we haven't seen yet. Use the example on the GeoPandas website where ```nepal_pop``` is dissolved by zone and population is summed with the ```aggfunc``` parameter: https://geopandas.org/en/stable/docs/user_guide/aggregation_with_dissolve.html

In [ ]:
state_pop = county_pop[['STNAME','POPULATION','geometry']].dissolve(by='STNAME',as_index=False,sort=False,aggfunc='sum')
state_pop

<div class="alert alert-info"> 

Programmatically show the state names for the states with the highest and lowest population.

**Hint:** Use ```.idxmax()``` and ```.idxmin()```.

In [ ]:
print(f'The most populous state is {state_pop.loc[state_pop.POPULATION.idxmax(),'STNAME']}.')
print(f'The least populous state is {state_pop.loc[state_pop.POPULATION.idxmin(),'STNAME']}.')

<div class="alert alert-info"> 

## F) Create buffers

Create a 15km buffer around each county population center in Mississippi and save the buffers to a new column in ```county_pop``` called BUFF_15KM.

In [ ]:
# add your code here
county_pop['BUFF_15KM'] = county_pop.loc[county_pop.STNAME=='Mississippi','POP_CENTER'].buffer(15000)
county_pop

<div class="alert alert-info"> 

## G) Find all buffers that intersect other states and visualize

### identify buffers with multi-state intersections

This task is a bit more complex than the others. The goal is to create a new column called MULTISTATE in ```county_pop``` that contains a value of True or False based on whether each 15km buffer intersects with any state that is not Mississippi. Write a custom function and use ```.apply()``` to accomplish this.

**Hints:**
- Use ```.apply()``` only on the rows of ```county_pop``` where there is a shape in the BUFF_15KM column. This means your apply should look something like ```county_pop.loc[conditional expression].apply(parameters)```. You can select the appropriate rows inside ```.loc[]``` with a combination of bitwise not and the ```.isna()``` function.
- In your custom function, test whether the buffer intersects each state shape in ```state_pop``` and sum that result. If a buffer intersects more than one state, your sum will be greater than 1 and your custom function should return True. If a buffer only intersects Mississippi, your sum will be 1 and your custom function should return False. You'll want to use an if-else statement to make your custom function return the appropriate result. 

In [ ]:
def buff_intersects_states(row,states):
    if row.BUFF_15KM.intersects(states.geometry).sum() > 1:
        return True
    else:
        return False
        
county_pop['MULTISTATE'] = county_pop.loc[~county_pop.BUFF_15KM.isna()].apply(buff_intersects_states,axis=1,args=(state_pop,))
county_pop

<div class="alert alert-info"> 

### visualize

Make a figure showing the county boundaries in black, the population centers whose 15km buffers are fully within Mississippi in blue, and the population centers whose 15km buffers extend beyond Mississippi in orange. Use ```lw=0.5``` for the county boundaries and ```marker='.'``` for all points.

In [ ]:
base = county_pop.plot(facecolor='none', edgecolor='black',lw=0.5)
county_pop.loc[county_pop.MULTISTATE==False,'POP_CENTER'].plot(ax=base, marker='.')
county_pop.loc[county_pop.MULTISTATE==True,'POP_CENTER'].plot(ax=base, marker='.')

# XV. At a Glance: Language Covered

The GeoPandas functionality that we covered at a glance...

## GeoPandas functions

```gpd.clip()```, ```gpd.GeooDataFrame()```, ```gpd.points_from_xy()```, ```gpd.read_file()```, ```gpd.sjoin()```, ```gpd.sjoin_nearest()```


## GeoPandas data structure (GeoDataFrame or GeoSeries) methods 

```.any()```, ```.apply()```, ```.buffer()```, ```.contains()```, ```.count_coordinates()```, ```.count_geometries()```, ```.dissolve()```, ```.distance()```,
```.get_coordinates()```, ```.intersects()```, ```.plot()```, ```.set_index()```, ```.to_crs()```, ```.union_all()```, ```.value_counts()```, ```.within()```

## GeoPandas data structure (DataFrame or Series) attributes

```.area```, ```.boundary```, ```.bounds```, ```.centroid```, ```.convex_hull```, ```.crs```, ```.envelope```, ```.total_bounds```


## Functions from other packages

```pyogrio.list_layers()```, and from matplotlib ```.set_ylim()```, ```.set_xlim()```


<div class="alert alert-success">

# XVI. Learning More About GeoPandas

For more about GeoPandas, start on the [GeoPandas website](https://geopandas.org/en/stable/index.html#) where you can find:

- The user guide, advanced user guide, api reference and small gallery of examples https://geopandas.org/en/stable/docs.html

The documentation on the GeoPandas website is less extensive than other packages we're learning in this course. There are additional tutoral materials on the web though, for example at DataCamp:
- DataCamp GeoPandas tutorial using hurricane data https://www.datacamp.com/tutorial/geospatial-data-python
- DataCamp GeoPandas tutorial with content similar to this lesson https://www.datacamp.com/tutorial/geopandas-tutorial-geospatial-analysis

And for asking the online community for help, the most relevant forums would probably be: 
- Stack Exchange https://stackexchange.com
- GIS Stack Exchange https://gis.stackexchange.com

</div>